# Quantum Circuit-Based Adaptation for Credit Risk Analysis
### A Hardware-Aware Variational Framework for the GCI Model — Project Overview

**Base Paper:** H. G. Ahmad et al., *"Quantum Circuit-Based Adaptation for Credit Risk Analysis,"* IEEE Transactions on Quantum Engineering, vol. 7, Art. no. 3103316, 2026. [DOI: 10.1109/TQE.2026.3691176](https://doi.org/10.1109/TQE.2026.3691176)

---

This notebook is the entry point of the repository. It lays out the **problem statement**, the **gaps in the base paper this project targets**, the **algorithmic framework**, the **project objectives**, and **what this project does differently from the base paper**, before the 9 stage-by-stage implementation notebooks begin.


## 1. Problem Statement

- **Financial Mandate:** Financial institutions must accurately evaluate Portfolio Credit Risk and Value at Risk (VaR) to protect capital frameworks against correlated borrower defaults under macro-economic shocks.

- **Classical Bottleneck:** Modeling joint default tails via the classical Gaussian Conditional-Independence (GCI) model relies on millions of Monte Carlo simulations, creating an exponential time bottleneck as the number of assets grows.

- **Theoretical QML Path:** Quantum Amplitude Estimation (QAE) provides a theoretical quadratic speedup, scaling at **O(√N)** compared to the classical **O(N)** operations required by Monte Carlo sampling.

- **Prefault-Tolerant Realities:** Modern NISQ processors suffer from severe readout flip errors, gate crosstalk, and limited qubit connectivity. Generic quantum compilations diverge from the target probability model without hardware-level corrections — this divergence is exactly what is investigated here.


## 2. Gaps Identified in the Base Paper

- **Algorithmic Idealism Gap:** Existing quantum credit-risk models (e.g., Egger et al. [1], Stamatopoulos et al. [19], Chakrabarti et al. [20]) evaluate architectures under the assumption of noiseless or perfectly error-corrected circuits, rendering them vulnerable when actually deployed on physical QPUs.

- **Offline Compilation Disconnect:** Standard layout/routing optimization steps (e.g., SABRE [38], which the base paper itself uses) focus strictly on minimizing structural gate count and SWAP depth. They optimize for *circuit structure*, not for *real-time gate error or noise behavior* — so a "well-transpiled" circuit can still produce a badly distorted output distribution.

- **In-Situ Invalidation of Angles:** Optimal rotation parameters calculated using classical, offline (noiseless) simulation fail on real processors due to per-qubit calibration decay, microwave pulse amplitude/phase drift, and local readout bias mismatches. The base paper's own results confirm this directly — θ₁ and θ₂ had to be **manually swept** against the real chip in fixed-degree steps (e.g., 21° → 1° grids, Section IV-A/IV-B) to recover a usable distribution, since the classically-trained angles did not transfer.

- **Manual, Non-Scalable Retuning:** The base paper's in-situ retuning procedure is a **brute-force grid sweep** performed by hand for each qubit pair/triplet — not an automated, repeatable optimization loop. This is explicitly called out in the paper as something that would need to be "repeated for each variable in play" as the model scales (Section V). This is the specific gap the novelty in Section 5 targets.


## 3. Algorithmic Framework

The project rests on four core algorithmic building blocks. Each gets its own deep-dive explanation notebook under `docs/concepts/`, but here is the map of how they fit together:

- **Variational Quantum Circuit (VQC Ansatz):** Serves as the structural template for a machine-learning-style circuit. It uses parameterized R<sub>y</sub>(θ) gates to control wavefunction amplitudes and CNOT gates to introduce asset/risk-factor correlations.

- **Gaussian Conditional-Independence (GCI) Model:** The core financial framework establishing the *target* distribution. The quantum computer's basis states (|00⟩ through |11⟩, or |000⟩ through |111⟩ for the 3-qubit case) are trained so their measured probabilities match this risk model's histogram.

- **Parameter-Shift Rule:** The quantum gradient-calculation engine. By shifting parameters by ±π/2 and re-measuring, it extracts *exact* analytical optimization gradients directly from circuit measurements — no classical backpropagation needed.

- **Adam Optimizer:** The classical gradient-descent engine. It consumes the gradients extracted via the parameter-shift rule and computes stable, adaptive parameter updates, iteratively converging the circuit's output toward the target Gaussian.


## 4. Project Objectives

- **Objective 1:** Implement a parameterized, hardware-efficient Variational Quantum Circuit (VQC Ansatz) on a simulator to model discrete 2-qubit and 3-qubit standard Gaussian curves.

- **Objective 2:** Construct a hybrid optimization engine linking the quantum circuit to a classical stochastic gradient descent routine (Adam Optimizer).

- **Objective 3:** Deploy the Parameter-Shift Rule directly within the circuit execution loop to extract analytical gradients without relying on classical backpropagation.

- **Objective 4:** Emulate realistic NISQ defects (readout bit-flips and depolarizing noise) and demonstrate how adaptive, in-situ gate updates successfully recover a high-fidelity Cumulative Distribution Function (CDF).


## 5. System Architecture & Novelty

The diagram below shows the full pipeline end to end, including where this project's core novelty sits.

![System Architecture](images/architecture_flowchart.png)

### What's novel here

The base paper's biggest self-acknowledged limitation (Section V) is that its in-situ retuning is a **manual, brute-force degree-by-degree grid sweep**, done by hand, per qubit pair, and explicitly *not* something that scales automatically as more risk factors/assets are added. That is the gap targeted here — algorithmically, not by claiming better hardware:

1. **Closed-Loop Auto-Calibration (replacing the manual grid sweep):** Instead of hand-sweeping θ in fixed-degree steps and eyeballing the best histogram, an **automated classical optimization loop** (SPSA / Bayesian Optimization) directly minimizes the Hellinger distance between the noisy-simulator output and the target distribution. The retuning that took a manual 21°→1° grid search in the base paper becomes a repeatable, scriptable function call.

2. **Scaling Beyond the Toy Model:** The base paper only *analytically projects* the resource cost of a 2-asset/2-risk-factor model (6 qubits) — it never actually builds or simulates one. This project implements and simulates that scaled circuit directly, empirically measuring how training difficulty, circuit depth, and noise sensitivity grow, rather than relying on linear extrapolation.

3. **Quantitative Noise-Mitigation Comparison:** The base paper applies noise correction only through in-situ angle retuning. This project additionally benchmarks standard **error-mitigation techniques available in Qiskit** — readout error mitigation and Zero-Noise Extrapolation (ZNE) — measuring their individual and combined effect on Hellinger fidelity and VaR accuracy, something the base paper explicitly lists as unimplemented future work.

4. **Noise-Aware Training (stretch goal):** Rather than the base paper's strict two-phase approach (train noiseless → retune on hardware after the fact), the noise model can also be injected *during* training itself, to test whether a circuit trained with noise-awareness from the start needs less post-hoc retuning.

> The result is an **automated, noise-aware calibration algorithm** that replaces manual grid-search retuning — evaluated on a noise-emulated simulator, with no claim of physical QPU deployment.


## 6. Reference

H. G. Ahmad, A. Sarno, M. El Bakraoui, C. Cosenza, C. Bésoin, F. Cibrario, V. Zaffaroni, G. Ranieri, R. Bertilone, V. Stasino, P. Mastrovito, F. Tafuri, D. Massarotti, L. Chabbra, and D. Corbelletto, "Quantum Circuit-Based Adaptation for Credit Risk Analysis," *IEEE Transactions on Quantum Engineering*, vol. 7, Art. no. 3103316, 2026, doi: [10.1109/TQE.2026.3691176](https://doi.org/10.1109/TQE.2026.3691176).

---
*Next: → `01_classical_gci_model.ipynb`*
